# Домашнее задание 3. Парсинг, Git и тестирование на Python

**Цели задания:**

* Освоить базовые подходы к web-scraping с библиотеками `requests` и `BeautisulSoup`: навигация по страницам, извлечение HTML-элементов, парсинг.
* Научиться автоматизировать задачи с использованием библиотеки `schedule`.
* Попрактиковаться в использовании Git и оформлении проектов на GitHub.
* Написать и запустить простые юнит-тесты с использованием `pytest`.


В этом домашнем задании вы разработаете систему для автоматического сбора данных о книгах с сайта [Books to Scrape](http://books.toscrape.com). Нужно реализовать функции для парсинга всех страниц сайта, извлечения информации о книгах, автоматического ежедневного запуска задачи и сохранения результата.

Важной частью задания станет оформление проекта: вы создадите репозиторий на GitHub, оформите `README.md`, добавите артефакты (код, данные, отчеты) и напишете базовые тесты на `pytest`.



In [ ]:
! pip install -q schedule pytest # установка библиотек, если ещё не

In [1]:
# Библиотеки, которые могут вам понадобиться
# При необходимости расширяйте список
import time
import json
from urllib.parse import urljoin
import requests
import schedule
from bs4 import BeautifulSoup

## Задание 1. Сбор данных об одной книге (2 баллов)

В этом задании мы начнем подготовку скрипта для парсинга информации о книгах со страниц каталога сайта [Books to Scrape](https://books.toscrape.com/).

Для начала реализуйте функцию `get_book_data`, которая будет получать данные о книге с одной страницы (например, с [этой](http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html)). Соберите всю информацию, включая название, цену, рейтинг, количество в наличии, описание и дополнительные характеристики из таблицы Product Information. Результат достаточно вернуть в виде словаря.

**Не забывайте про соблюдение PEP-8** — помимо качественно написанного кода важно также документировать функции по стандарту:
* кратко описать, что она делает и для чего нужна;
* какие входные аргументы принимает, какого они типа и что означают по смыслу;
* аналогично описать возвращаемые значения.

*P. S. Состав, количество аргументов функции и тип возвращаемого значения можете менять как вам удобно. То, что написано ниже в шаблоне — лишь пример.*

In [2]:
def get_book_data(book_url: str) -> dict:
    """Функция парсинга одной книги
    Args:
        book_url: str - электронный адрес
    Returns: 
        book_data: dict - словарь метаданных книги
    """

    response = requests.get(book_url)
    response.encoding = 'utf-8'
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    title = soup.find("h1").get_text(strip=True)

    price_tag = soup.find("p", class_="price_color")
    price = price_tag.get_text(strip=True) if price_tag else "Не найдено"

    rating_tag = soup.find("p", class_="star-rating")
    rating = rating_tag["class"][1] if rating_tag and len(rating_tag["class"]) > 1 else "Не найдено"
  
    availability_tag = soup.find("p", class_="instock availability")
    availability_text = availability_tag.get_text(strip=True) if availability_tag else ""
    quantity_in_stock = availability_text

    meta_desc = soup.find("meta", attrs={"name": "description"})
    description = meta_desc["content"].strip() if meta_desc and meta_desc.has_attr("content") else "Описание не найдено"

    product_info = {}
    table = soup.find("table", class_="table table-striped")
    if table:
        rows = table.find_all("tr")
        for row in rows:
            th = row.find("th")
            td = row.find("td")
            if th and td:
                key = th.get_text(strip=True)
                value = td.get_text(strip=True)
                product_info[key] = value

    book_data = {
        "Название": title,
        "Цена": price,
        "Рейтинг": rating,
        "Наличие": quantity_in_stock,
        "Описание": description,
    }
    book_data.update(product_info)

    return book_data

In [3]:
# Используйте для самопроверки
book_url = 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'
get_book_data(book_url)

{'Название': 'A Light in the Attic',
 'Цена': '£51.77',
 'Рейтинг': 'Three',
 'Наличие': 'In stock (22 available)',
 'Описание': "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a treetopIs no safe place

## Задание 2. Сбор данных обо всех книгах (2 баллов)

Создайте функцию `scrape_books`, которая будет проходиться по всем страницам из каталога (вида `http://books.toscrape.com/catalogue/page-{N}.html`) и осуществлять парсинг всех страниц в цикле, используя ранее написанную `get_book_data`.

Добавьте аргумент-флаг, который будет отвечать за сохранение результата в файл: если он будет равен `True`, то информация сохранится в ту же папку в файл `books_data.txt`; иначе шаг сохранения будет пропущен.

**Также не забывайте про соблюдение PEP-8**

In [ ]:
def scrape_books(current_url: str, is_save: bool = True) -> dict:
    """Функция парсинга каталога книг
    Args:
        current_url: str - электронный адрес страницы
        is_save: bool - флаг необходимости сохранения информации в текстовый файл, по умолчанию True
    Returns: 
        books_data: dict - словарь c информацией обо всех книгах каталога
    """

    books_data = {}
    book_counter = 0

    while current_url:
      print(f'Обрабатываю страницу каталога: {current_url}')
      response = requests.get(current_url)
      response.encoding = 'utf-8'
      soup = BeautifulSoup(response.text, 'html.parser')

      # находим все карточки книг на текущей странице каталога
      articles = soup.find_all('article', class_='product_pod')

      for article in articles:
        # Получаем относительную ссылку на страницу книги и превращаем её в полную
        relative_book_url = article.find('h3').find('a')['href']
        book_url = urljoin(current_url, relative_book_url)

        # добавляем информацию по одной книге в словарь
        book_counter += 1
        book_info = get_book_data(book_url) 
        books_data[f'book_{book_counter}'] = book_info

      # переходим на следующую страницу каталога
      next_button = soup.find('li', class_='next')
      if next_button:
        next_page_relative = next_button.find('a')['href']
        current_url = urljoin(current_url, next_page_relative)
      else:
        current_url = None
    
    if is_save:
      with open("books_data.txt", "w", encoding="utf-8") as file:
        json.dump(books_data, file, ensure_ascii=False, indent=4)
      
    return books_data

In [5]:
# Проверка работоспособности функции
current_url = 'http://books.toscrape.com/catalogue/category/books_1/index.html'
res = scrape_books(current_url, is_save=True) # Допишите ваши аргументы
print(type(res), len(res)) # и проверки

Обрабатываю страницу каталога: http://books.toscrape.com/catalogue/category/books_1/index.html
Обрабатываю страницу каталога: http://books.toscrape.com/catalogue/category/books_1/page-2.html
Обрабатываю страницу каталога: http://books.toscrape.com/catalogue/category/books_1/page-3.html
Обрабатываю страницу каталога: http://books.toscrape.com/catalogue/category/books_1/page-4.html
Обрабатываю страницу каталога: http://books.toscrape.com/catalogue/category/books_1/page-5.html
Обрабатываю страницу каталога: http://books.toscrape.com/catalogue/category/books_1/page-6.html
Обрабатываю страницу каталога: http://books.toscrape.com/catalogue/category/books_1/page-7.html
Обрабатываю страницу каталога: http://books.toscrape.com/catalogue/category/books_1/page-8.html
Обрабатываю страницу каталога: http://books.toscrape.com/catalogue/category/books_1/page-9.html
Обрабатываю страницу каталога: http://books.toscrape.com/catalogue/category/books_1/page-10.html
Обрабатываю страницу каталога: http://bo

## Задание 3. Настройка регулярной выгрузки (1 баллов)

Настройте автоматический запуск функции сбора данных каждый день в 19:00.
Для автоматизации используйте библиотеку `schedule`. Функция должна запускаться в указанное время и сохранять обновленные данные в текстовый файл.



Бесконечный цикл должен обеспечивать постоянное ожидание времени для запуска задачи и выполнять ее по расписанию. Однако чтобы не перегружать систему, стоит подумать о том, чтобы выполнять проверку нужного времени не постоянно, а раз в какой-то промежуток. В этом вам может помочь `time.sleep(...)`.

Проверьте работоспособность кода локально на любом времени чч:мм.



In [ ]:
# НАЧАЛО ВАШЕГО РЕШЕНИЯ
schedule.every().day.at("19:00").do(scrape_books, current_url=current_url)
print("Планировщик запущен.")
while True:
    schedule.run_pending()
    time.sleep(30)
# КОНЕЦ ВАШЕГО РЕШЕНИЯ

Планировщик запущен.


In [ ]:
# проверка на другом времени
current_url = 'http://books.toscrape.com/catalogue/category/books_1/page-49.html'
schedule.every().day.at("21:53").do(scrape_books, current_url=current_url)
print("Планировщик запущен.")
while True:
    schedule.run_pending()
    time.sleep(30)

Планировщик запущен.
Обрабатываю страницу каталога: http://books.toscrape.com/catalogue/category/books_1/page-49.html
Обрабатываю страницу каталога: http://books.toscrape.com/catalogue/category/books_1/page-50.html


## Задание 4. Написание автотестов (1,5 баллов)

Создайте минимум три автотеста для ключевых функций парсинга — например, `get_book_data` и `scrape_books`. Идеи проверок (можете использовать свои):

* данные о книге возвращаются в виде словаря с нужными ключами;
* список ссылок или количество собранных книг соответствует ожиданиям;
* значения отдельных полей (например, `title`) корректны.

Оформите тесты в отдельном скрипте `tests/test_scraper.py`, используйте библиотеку `pytest`. Убедитесь, что тесты проходят успешно при запуске из терминала командой `pytest`.

Также выведите результат их выполнения в ячейке ниже.

**Не забывайте про соблюдение PEP-8**


In [6]:
# Ячейка для демонстрации работоспособности
# Сам код напишите в отдельном скрипте
! pytest test/test_scraper.py

"pytest" �� ���� ����७��� ��� ���譥�
��������, �ᯮ��塞�� �ணࠬ��� ��� ������ 䠩���.


## Задание 5. Оформление проекта на GitHub и работа с Git (3,5 баллов)

В этом задании нужно воспользоваться системой контроля версий Git и платформой GitHub для хранения и управления своим проектом. **Ссылку на свой репозиторий пришлите в форме для сдачи ответа.**

### Пошаговая инструкция и задания

**1. Установите Git на свой компьютер.**

* Для Windows: [скачайте установщик](https://git-scm.com/downloads) и выполните установку.
* Для macOS:

  ```
  brew install git
  ```
* Для Linux:

  ```
  sudo apt update
  sudo apt install git
  ```

**2. Настройте имя пользователя и email.**

Это нужно для подписи ваших коммитов, сделайте в терминале через `git config ...`.

**3. Создайте аккаунт на GitHub**, если у вас его еще нет:
[https://github.com](https://github.com)

**4. Создайте новый репозиторий на GitHub:**

* Найдите кнопку **New repository**.
* Укажите название, краткое описание, выберите тип **Public** (чтобы мы могли проверить ДЗ).
* Не ставьте галочку Initialize this repository with a README.

**5. Создайте локальную папку с проектом.** Можно в терминале, можно через UI, это не имеет значения.

**6. Инициализируйте Git в этой папке.** Здесь уже придется воспользоваться некоторой командой в терминале.

**7. Привяжите локальный репозиторий к удаленному на GitHub.**

**8. Создайте ветку разработки.** По умолчанию вы будете находиться в ветке `main`, создайте и переключитесь на ветку `hw-books-parser`.

**9. Добавьте в проект следующие файлы и папки:**

* `scraper.py` — ваш основной скрипт для сбора данных.
* `README.md` — файл с кратким описанием проекта:

  * цель;
  * инструкции по запуску;
  * список используемых библиотек.
* `requirements.txt` — файл со списком зависимостей, необходимых для проекта (не присылайте все из глобального окружения, создайте изолированную виртуальную среду, добавьте в нее все нужное для проекта и получите список библиотек через `pip freeze`).
* `artifacts/` — папка с результатами парсинга (`books_data.txt` — полностью или его часть, если весь не поместится на GitHub).
* `notebooks/` — папка с заполненным ноутбуком `HW_03_python_ds.ipynb` и запущенными ячейками с выводами на экран.
* `tests/` — папка с тестами на `pytest`, оформите их в формате скрипта(-ов) с расширением `.py`.
* `.gitignore` — стандартный файл, который позволит исключить временные файлы при добавлении в отслеживаемые (например, `__pycache__/`, `.DS_Store`, `*.pyc`, `venv/` и др.).


**10. Сделайте коммит.**

**11. Отправьте свою ветку на GitHub.**

**12. Создайте Pull Request:**

* Перейдите в репозиторий на GitHub.
* Нажмите кнопку **Compare & pull request**.
* Укажите, что было добавлено, и нажмите **Create pull request**.

**13. Выполните слияние Pull Request:**

* Убедитесь, что нет конфликтов.
* Нажмите **Merge pull request**, затем **Confirm merge**.

**14. Скачайте изменения из основной ветки локально.**



### Требования к итоговому репозиторию

* Файл `scraper.py` с рабочим кодом парсера.
* `README.md` с описанием проекта и инструкцией по запуску.
* Папка `artifacts/` с результатом сбора данных (`.txt` файл).
* Папка `tests/` с тестами на `pytest`.
* Папка `notebooks/` с заполненным ноутбуком `HW_03_python_ds_2025.ipynb`.
* Pull Request с комментарием из ветки `hw-books-parser` в ветку `main`.
* Примерная структура:

  ```
  books_scraper/
  ├── artifacts/
  │   └── books_data.txt
  ├── notebooks/
  │   └── HW_03_python_ds_2025.ipynb
  ├── scraper.py
  ├── README.md
  ├── tests/
  │   └── test_scraper.py
  ├── .gitignore
  └── requirements.txt
  ```

## Выводы, рефлексия и сдача

### Рефлексия после выполнения (заполните здесь)

**1. Что было задано и что получилось**:

`Ответ: По заданию необходимо было создать парсер книжного сайта, который тянет подробную информацию по книгам и запускается в определенное время. Так же необходимо было создать несколько автотестов, для отслеживания корректной работы. Еще одной задачей было создание, правильное оформление и наполнения репозитория на Github.`

**2. Как выполняли**: что сработало сразу, где застряли и как выбрались:

`Ответ: Задача 1. Было не очевидно, что данные по цене, количеству штук и ревью это таблица <table>, и <th> это заголовок ячейки таблицы, а <td> - данные таблицы. Так же очень помогла команда ctrl+u в браузере и клавиша F12. Возникли трудности с определением рейтинга книги и тем, что рейтнг прописывется словом. Задача 2. Сложность с организацией перехода на следующую страницу каталога и как записать словарь в .txt формат. Задача 4. Не сразу понял, что в итоге требовалось по заданию. Рефлексию по первому заданию писал до занятия по BeatifulSoup. В ноутбуке по занятию вопрос с таблицей отчасти был затронут. `

**3. Использование ИИ**: цель, использованные промпты, где помощник ошибся и как вы это поняли. Если не использовали, так и напишите:

`Ответ: Задача 1. Как записать список в столбик?. Задача 2. Как организовать переход к новой странице при парсинге сайтов? Как записать словарь в .txt файл? Задача 4. Задавл много вопросов по работе с фреймворком pytest`

**4. Что осталось непонятным** (честный список, он поможет на консультации):

`Ответ: Не сразу понял задание с автотестами, что в итоге необходимо было сделать. `

**5. Самооценка по критериям задания**:

`Ответ: На текущий момент самое сложное домашнее задание. Практически все библиотеки, используемые в задании, для меня были новыми, опыта работы в них не было. Так же для выполнения задания часто приходилось обращаться к ИИ. Так же впревые работал с git.`

**6. Дополнительный вопрос: обратная связь по заданию.** Что было ценно, что сложно, а что, на ваш взгляд, требует доработки?

`Ответ: Самое интересное на текущий момент домашнее задание. Задания были и интересные и сложные. Так же в ходе работы появился ряд идей, как полученные навыки можно в дальнейшем использовать. `

> Напоминание: рефлексия пишется своими словами, без помощи ИИ.

